# GNN Encoder Sweep Results

plots the CSVs `scripts/gnn_encoder_sweep.py` writes to `outputs/encoders/GNN/sweep/`, so run the sweep first. this only reads them, so it takes a few seconds

configs are ranked by mean val RMSE across seeds, error bars are std across seeds. test only shows up for the finalists, since that's the only place it gets used

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)

SWEEP = Path("../outputs/encoders/GNN/sweep")
stages = {p.stem: pd.read_csv(p) for p in sorted(SWEEP.glob("*.csv"))}

CONFIG_KEYS = [
    "HIDDEN_DIMS", "DROPOUT", "LR", "WEIGHT_DECAY", "BATCH_SIZE", "EPOCHS", "OPT",
]
SHORT = {
    "HIDDEN_DIMS": "dims", "DROPOUT": "drop", "LR": "lr", "WEIGHT_DECAY": "wd", 
    "BATCH_SIZE": "batch", "EPOCHS": "epochs", "OPT": "opt",
}
METRICS = ["train_RMSE", "val_RMSE", "test_RMSE", "test_R2", "test_ROC-AUC", "best_epoch"]

{name: len(df) for name, df in stages.items()}

In [ ]:
# config keys that differ between rows
def varying(df):
    return [k for k in CONFIG_KEYS if k in df and df[k].nunique(dropna=False) > 1]


def with_labels(df):
    keys = varying(df)
    labels = df.groupby("cfg")[keys].first().apply(
        lambda r: ", ".join(f"{SHORT[k]}={v}" for k, v in r.items() if not pd.isna(v)),
        axis=1,
    )
    return df.assign(config=df["cfg"].map(labels))


# one row per config: mean over seeds + the val spread
def summarize(df):
    g = (df if "config" in df else with_labels(df)).groupby("config")
    out = g[[m for m in METRICS if m in df]].mean()
    out["val_RMSE_std"] = g["val_RMSE"].std()
    out["seeds"] = g.size()
    return out.sort_values("val_RMSE")


def show(name, n=10):
    s = summarize(stages[name]).head(n)
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(s) + 1.2))
    ax.errorbar(s["val_RMSE"], range(len(s)), xerr=s["val_RMSE_std"], fmt="o", capsize=3)
    ax.set(yticks=range(len(s)), yticklabels=s.index, xlabel="val RMSE", title=f"{name} (top {len(s)})")
    ax.invert_yaxis()
    plt.show()
    return s.round(3)

## 1-architecture

In [ ]:
show("1-architecture")

## 2-regularization, 3-training-dynamics, 4-optimizer, 5-holdout-seeds

In [ ]:
s2 = stages["2-regularization"]

grid = s2.pivot_table(index="DROPOUT", columns="WEIGHT_DECAY", values="val_RMSE", aggfunc="mean")

fig, ax = plt.subplots(figsize=(7, 5))

im = ax.imshow( grid, cmap="viridis_r")

for (r, c), v in np.ndenumerate(grid.to_numpy()):
    ax.text( c, r, f"{v:.3f}", ha="center", va="center", color="k" if im.norm(v) < 0.5 else "w")

ax.set_xticks(range(grid.shape[1]), [f"{w:g}" for w in grid.columns])

ax.set_yticks(range(grid.shape[0]), grid.index)

ax.set(xlabel="Weight decay", ylabel="Dropout", title="Regularization sweep",)

fig.colorbar(im, ax=ax, label="Mean validation RMSE (lower is better)")

plt.show()

show("2-regularization")

In [ ]:
show("3-training-dynamics")

In [ ]:
show("4-optimizer")

## 5-holdout-seeds

In [ ]:
summarize(stages["5-holdout-seeds"]).round(3)

In [ ]:
show("6-final-check")